# 02 - Clean: build a modelling-ready table

**Input:** `data/raw/diabetic_data.csv` (never modified).
**Output:** `data/processed/diabetic_clean.csv`: one row per encounter, no missing values, binary target `readmit_30`.

Every cleaning decision below has a one-line reason. In an interview, the *reason* matters more than the code.

| # | Decision | Reason |
|---|---|---|
| 1 | Remove encounters ending in death or hospice | They cannot (or rarely) come back for reasons unrelated to care quality |
| 2 | Remove 3 rows with gender `Unknown/Invalid` | Too few to learn from; would break the gender fairness check |
| 3 | Target: `<30` = 1, else 0; drop raw `readmitted` | Our question is 30-day readmission; the raw column would leak the answer |
| 4 | Drop `weight` and `payer_code` | 97% / 40% missing; payer is an insurance (income) proxy we don't want driving care decisions |
| 5 | Missing `race`, `medical_specialty` -> `"Unknown"` | Keep the rows; "unknown" can itself carry signal |
| 6 | Group ~700 ICD-9 diagnosis codes into 9 disease groups | Too many rare codes to learn from individually |
| 7 | Decode ID columns to labels; merge NULL/Not Mapped into `"Unknown"`; rare -> `"Other"` | IDs are labels, not numbers |
| 8 | Lab test `"None"` -> `"NotTested"` | Make the meaning explicit and stop pandas reading it as missing later |
| 9 | Drop near-constant drug columns (>= 99% "No") | Almost no variation = almost no information |
| 10 | Yes/No columns -> 1/0; add numeric `age_num` | Simple, model-ready encodings |

**Repeat encounters:** we keep *all* encounters here. Leakage is handled in notebook 03 by splitting **by patient**
(the same `patient_nbr` never appears in both train and test).

## 1. Setup and load (same settings as notebook 01)

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = PROJECT / "data" / "raw"
PROCESSED = PROJECT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

ID_COLS = ["encounter_id", "patient_nbr",
           "admission_type_id", "discharge_disposition_id", "admission_source_id"]

df = pd.read_csv(
    RAW / "diabetic_data.csv",
    na_values=["?"],
    keep_default_na=False,
    dtype={c: str for c in ID_COLS},
    low_memory=False,
)

steps = []                      # a log of how many rows survive each step
def log(step):
    steps.append({"step": step, "rows": len(df), "cols": df.shape[1]})

log("raw")
df.shape

(101766, 50)

Load the ID mappings (same parsing as notebook 01). `.strip()` removes stray spaces, because some descriptions in the file
start with a space (e.g. `" Not Mapped"`), which would otherwise not match `"Not Mapped"`.

In [2]:
raw_map = pd.read_csv(RAW / "IDS_mapping.csv", header=None, dtype=str, keep_default_na=False)

mappings = {}
current = None
for code_value, description in raw_map.itertuples(index=False):
    if code_value.endswith("_id"):
        current = code_value
        mappings[current] = {}
    elif code_value != "":
        mappings[current][code_value] = description.strip()

{k: len(v) for k, v in mappings.items()}

{'admission_type_id': 8,
 'discharge_disposition_id': 30,
 'admission_source_id': 25}

## 2. Remove death and hospice encounters
Discharge codes 11, 19, 20, 21 = expired; 13, 14 = hospice. We list them from the mapping file so the choice is visible,
not hard-coded from memory.

In [3]:
dd_map = mappings["discharge_disposition_id"]
END_OF_LIFE = [c for c, d in dd_map.items() if "expired" in d.lower() or "hospice" in d.lower()]
for c in END_OF_LIFE:
    print(c, "-", dd_map[c])

mask = df["discharge_disposition_id"].isin(END_OF_LIFE)
print(f"\nRemoving {mask.sum():,} encounters")
df = df[~mask].copy()
log("drop expired/hospice")

11 - Expired
13 - Hospice / home
14 - Hospice / medical facility
19 - Expired at home. Medicaid only, hospice.
20 - Expired in a medical facility. Medicaid only, hospice.
21 - Expired, place unknown. Medicaid only, hospice.

Removing 2,423 encounters


## 3. Remove invalid gender rows

In [4]:
print(df["gender"].value_counts())
df = df[df["gender"] != "Unknown/Invalid"].copy()
log("drop gender Unknown/Invalid")

gender
Female             53454
Male               45886
Unknown/Invalid        3
Name: count, dtype: int64


## 4. Binary target
`(condition).astype(int)` turns True/False into 1/0. We then drop the raw `readmitted` column:
if it stayed, a model could read `">30"` vs `"NO"` and partly see the answer.

In [5]:
df["readmit_30"] = (df["readmitted"] == "<30").astype(int)
df = df.drop(columns="readmitted")
print(f"Base rate after removals: {df['readmit_30'].mean():.2%}")
log("binary target")

Base rate after removals: 11.39%


## 5. Drop high-missing / sensitive columns

In [6]:
print(df[["weight", "payer_code"]].isna().mean().round(3))
df = df.drop(columns=["weight", "payer_code"])
log("drop weight, payer_code")

weight        0.969
payer_code    0.397
dtype: float64


## 6. Fill missing categories with "Unknown", group rare specialties
`group_rare` keeps categories that cover at least 1% of rows and relabels the rest as `"Other"`.
Rare categories make one-hot encoding produce many near-empty columns that a model can overfit on.

In [7]:
def group_rare(s, min_share=0.01, other="Other"):
    share = s.value_counts(normalize=True)
    keep = share[share >= min_share].index
    return s.where(s.isin(keep), other)

df["race"] = df["race"].fillna("Unknown")
df["medical_specialty"] = group_rare(df["medical_specialty"].fillna("Unknown"))

print(df["race"].value_counts(), "\n")
print(df["medical_specialty"].value_counts())

race
Caucasian          74220
AfricanAmerican    18772
Unknown             2232
Hispanic            2017
Other               1471
Asian                628
Name: count, dtype: int64 

medical_specialty
Unknown                       48614
InternalMedicine              14237
Other                          8199
Emergency/Trauma               7419
Family/GeneralPractice         7252
Cardiology                     5278
Surgery-General                3059
Nephrology                     1539
Orthopedics                    1392
Orthopedics-Reconstructive     1230
Radiologist                    1121
Name: count, dtype: int64


## 7. Group diagnosis codes (ICD-9)
`diag_1` is the primary diagnosis, `diag_2`/`diag_3` secondary. Codes are ICD-9: numbers like `428` (heart failure),
`250.83` (diabetes), or codes starting with `V`/`E` (supplementary factors / external causes).

We use the grouping from the paper that published this dataset (Strack et al., 2014):

| Group | ICD-9 range |
|---|---|
| Circulatory | 390-459, 785 |
| Respiratory | 460-519, 786 |
| Digestive | 520-579, 787 |
| Diabetes | 250.xx |
| Injury | 800-999 |
| Musculoskeletal | 710-739 |
| Genitourinary | 580-629, 788 |
| Neoplasms | 140-239 |
| Other | everything else, incl. V and E codes |

In [8]:
def icd9_group(code):
    if pd.isna(code):
        return "Missing"
    if code.startswith(("V", "E")):
        return "Other"
    if code.startswith("250"):
        return "Diabetes"
    n = float(code)
    if 390 <= n < 460 or int(n) == 785: return "Circulatory"
    if 460 <= n < 520 or int(n) == 786: return "Respiratory"
    if 520 <= n < 580 or int(n) == 787: return "Digestive"
    if 800 <= n < 1000:                 return "Injury"
    if 710 <= n < 740:                  return "Musculoskeletal"
    if 580 <= n < 630 or int(n) == 788: return "Genitourinary"
    if 140 <= n < 240:                  return "Neoplasms"
    return "Other"

# quick self-test on known codes before applying it to 100k rows
tests = {"428": "Circulatory", "250.83": "Diabetes", "486": "Respiratory", "V57": "Other",
         "820": "Injury", "715": "Musculoskeletal", "599": "Genitourinary", "174": "Neoplasms",
         "785": "Circulatory", "780": "Other"}
assert all(icd9_group(c) == g for c, g in tests.items()), "icd9_group self-test failed"

for col in ["diag_1", "diag_2", "diag_3"]:
    df[col] = df[col].map(icd9_group)

df["diag_1"].value_counts()

diag_1
Circulatory        29680
Other              17793
Respiratory        13934
Digestive           9333
Diabetes            8661
Injury              6851
Genitourinary       5002
Musculoskeletal     4935
Neoplasms           3131
Missing               20
Name: count, dtype: int64

## 8. Decode the admission/discharge ID columns
Map each code to its text label, merge the "we don't know" labels into one `"Unknown"`, then group rare labels.

In [9]:
UNKNOWN_LABELS = {"NULL", "Not Mapped", "Not Available", "Unknown/Invalid"}

for col in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
    label = df[col].map(mappings[col]).fillna("Unknown")
    label = label.where(~label.isin(UNKNOWN_LABELS), "Unknown")
    new_col = col.replace("_id", "")
    df[new_col] = group_rare(label)
    df = df.drop(columns=col)
    print(df[new_col].value_counts(), "\n")

log("decode ID columns")

admission_type
Emergency    52369
Elective     18667
Urgent       18132
Unknown      10144
Other           28
Name: count, dtype: int64 

discharge_disposition
Discharged to home                                                                   60232
Discharged/transferred to SNF                                                        13954
Discharged/transferred to home with home health service                              12902
Unknown                                                                               4680
Other                                                                                 2268
Discharged/transferred to another short term hospital                                 2128
Discharged/transferred to another rehab fac including rehab units of a hospital .     1992
Discharged/transferred to another type of inpatient care institution                  1184
Name: count, dtype: int64 

admission_source
Emergency Room                                55848
Physician Refer

## 9. Lab tests: make "not tested" explicit

In [10]:
for col in ["max_glu_serum", "A1Cresult"]:
    df[col] = df[col].replace("None", "NotTested")
    print(df[col].value_counts(), "\n")

max_glu_serum
NotTested    94188
Norm          2545
>200          1419
>300          1188
Name: count, dtype: int64 

A1Cresult
NotTested    82506
>8            8137
Norm          4922
>7            3775
Name: count, dtype: int64 



## 10. Diabetes medication columns
Each drug column says `No` (not prescribed), `Steady`, `Up` (dose increased) or `Down` (dose decreased).
We drop drugs where at least 99% of rows are `No`.

In [11]:
DRUGS = ["metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride", "acetohexamide",
         "glipizide", "glyburide", "tolbutamide", "pioglitazone", "rosiglitazone", "acarbose", "miglitol",
         "troglitazone", "tolazamide", "examide", "citoglipton", "insulin", "glyburide-metformin",
         "glipizide-metformin", "glimepiride-pioglitazone", "metformin-rosiglitazone", "metformin-pioglitazone"]

share_no = (df[DRUGS] == "No").mean().sort_values()
drop_drugs = share_no[share_no >= 0.99].index.tolist()
keep_drugs = share_no[share_no < 0.99].index.tolist()
print("Keeping:", keep_drugs)
print("Dropping:", drop_drugs)
df = df.drop(columns=drop_drugs)
log("drop near-constant drugs")

Keeping: ['insulin', 'metformin', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'glimepiride', 'repaglinide']
Dropping: ['glyburide-metformin', 'nateglinide', 'acarbose', 'chlorpropamide', 'tolazamide', 'miglitol', 'tolbutamide', 'glipizide-metformin', 'troglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'acetohexamide', 'glimepiride-pioglitazone', 'examide', 'citoglipton']


## 11. Simple encodings
- `change` (`Ch`/`No`) and `diabetesMed` (`Yes`/`No`) -> 1/0.
- `age` is a 10-year band like `[70-80)`. We keep the band (useful for the fairness check) and add `age_num` = band midpoint (75),
  because age has a natural order a model should be able to use.

In [12]:
df["change"] = (df["change"] == "Ch").astype(int)
df["diabetesMed"] = (df["diabetesMed"] == "Yes").astype(int)

df["age_num"] = df["age"].str.extract(r"\[(\d+)-")[0].astype(int) + 5
df[["age", "age_num"]].drop_duplicates().sort_values("age_num")

,age,age_num
0,[0-10),5
1,[10-20),15
2,[20-30),25
3,[30-40),35
4,[40-50),45
5,[50-60),55
6,[60-70),65
7,[70-80),75
8,[80-90),85
9,[90-100),95


## 12. Final checks
`assert` stops the notebook with an error if a condition is false. These are cheap guards against silent mistakes.

In [13]:
assert df.isna().sum().sum() == 0, "there are still missing values"
assert df["encounter_id"].is_unique, "duplicate encounters"
assert set(df["readmit_30"].unique()) == {0, 1}

print(f"Rows: {len(df):,}   Columns: {df.shape[1]}")
print(f"Unique patients: {df['patient_nbr'].nunique():,}")
print(f"Base rate (readmit_30): {df['readmit_30'].mean():.2%}")
pd.DataFrame(steps)

Rows: 99,340   Columns: 34
Unique patients: 69,987
Base rate (readmit_30): 11.39%


,step,rows,cols
0,raw,101766,50
1,drop expired/hospice,99343,50
2,drop gender Unknown/Invalid,99340,50
3,binary target,99340,50
4,"drop weight, payer_code",99340,48
5,decode ID columns,99340,48
6,drop near-constant drugs,99340,33


In [14]:
df.dtypes.to_frame("dtype").T

,encounter_id,patient_nbr,race,gender,age,time_in_hospital,medical_specialty,num_lab_procedures,num_procedures,num_medications,...,pioglitazone,rosiglitazone,insulin,change,diabetesMed,readmit_30,admission_type,discharge_disposition,admission_source,age_num
dtype,str,str,str,str,str,int64,str,int64,int64,int64,...,str,str,str,int64,int64,int64,str,str,str,int64


## 13. Save
CSV does not remember data types, so notebook 03 must again read the two ID columns as strings.

In [15]:
out = PROCESSED / "diabetic_clean.csv"
df.to_csv(out, index=False)
print("Saved", out, f"({out.stat().st_size / 1e6:.1f} MB)")

Saved c:\Projects\readmission\data\processed\diabetic_clean.csv (21.1 MB)
